# 🩺 MedOneX — Telegram Group Content Sync
### Uploads all module content to Group Topics automatically

**What this does:**
1. Fetches all Modules → Subjects → Lectures from MedOneX
2. Creates a Telegram Forum Topic for each Module
3. Downloads all files, PDFs, videos for each lecture
4. Uploads them directly to the correct topic (not just links)
5. Organizes content: Subject header → Lecture → Files/Videos

**Requirements:**
- Bot must be **Admin** in the Supergroup
- Group must have **Topics (Forum)** enabled
- Fill the config in Cell 2 before running

In [ ]:
# ─── Cell 1: Install Dependencies ───────────────────────────────────────────
!pip install requests tqdm yt-dlp -q
print('✅ Dependencies ready.')

In [ ]:
# ─── Cell 2: ⚙️ CONFIGURATION — Fill these before running ────────────────────

BOT_TOKEN    = "8730612796:AAEgChDIg9OU7-uSTffYeb_sjsE5kXGg9eY"  # Your bot token
GROUP_ID     = -1002000000000        # Your supergroup ID (MUST be negative, e.g. -1002123456789)
API_BASE_URL = "https://medonex.vercel.app"                       # Platform URL
SEMESTER_ID  = "PASTE_YOUR_SEMESTER_ID_HERE"                      # Target semester

# Optional: If you want to skip certain modules by name
SKIP_MODULES = []   # e.g. ['Anatomy', 'Physiology']

# ── Advanced ──
RATE_DELAY   = 1.2   # seconds between Telegram API calls
MAX_FILE_MB  = 49    # files larger than this are sent as links

import requests, os, re, time, glob, subprocess
from pathlib import Path
from tqdm import tqdm

WORK_DIR = '/content/medonex_sync'
os.makedirs(WORK_DIR, exist_ok=True)
TG = f'https://api.telegram.org/bot{BOT_TOKEN}'

print('✅ Config loaded.')
print(f'   Bot token: {BOT_TOKEN[:20]}...')
print(f'   Group ID : {GROUP_ID}')
print(f'   Platform : {API_BASE_URL}')
print(f'   Semester : {SEMESTER_ID}')

In [ ]:
# ─── Cell 3: Telegram API Helper ─────────────────────────────────────────────

def tg(method, data=None, files=None, retry=4):
    """Call Telegram Bot API — auto-retries on rate limits."""
    if data is None: data = {}
    for attempt in range(retry):
        try:
            if files:
                r = requests.post(f'{TG}/{method}', data=data, files=files, timeout=180)
            else:
                r = requests.post(f'{TG}/{method}', json=data, timeout=30)
            result = r.json()
            if result.get('ok'):
                return result['result']
            if result.get('error_code') == 429:
                wait = result.get('parameters', {}).get('retry_after', 15)
                print(f'  ⏳ Rate limited, waiting {wait}s...')
                time.sleep(wait + 1)
                continue
            print(f'  ⚠️ [{method}]: {result.get("description", "Unknown error")}')
            return None
        except Exception as e:
            print(f'  ❌ Request error: {e}')
            time.sleep(3)
    return None

def create_topic(name, color=0x6FB9F0):
    result = tg('createForumTopic', {'chat_id': GROUP_ID, 'name': name[:128], 'icon_color': color})
    if result:
        tid = result['message_thread_id']
        print(f'  📌 Topic created: "{name}" → thread_id={tid}')
        return tid
    return None

def send_text(thread_id, text, parse_mode='Markdown'):
    time.sleep(RATE_DELAY)
    return tg('sendMessage', {
        'chat_id': GROUP_ID, 'message_thread_id': thread_id,
        'text': text[:4096], 'parse_mode': parse_mode,
        'disable_web_page_preview': True
    })

def send_file(thread_id, path, caption='', as_video=False):
    time.sleep(RATE_DELAY)
    size_mb = os.path.getsize(path) / 1024 / 1024
    if size_mb > MAX_FILE_MB:
        print(f'    ⚠️ File too large ({size_mb:.1f}MB) — will send as link')
        return None
    method = 'sendVideo' if as_video else 'sendDocument'
    key    = 'video'    if as_video else 'document'
    with open(path, 'rb') as f:
        return tg(method,
                  {'chat_id': GROUP_ID, 'message_thread_id': thread_id, 'caption': caption[:1024]},
                  {key: f})

print('✅ Telegram helpers ready.')

In [ ]:
# ─── Cell 4: Download Helpers ─────────────────────────────────────────────────

def download_direct(url, dest):
    """Direct HTTP download."""
    try:
        r = requests.get(url, stream=True, timeout=120,
                         headers={'User-Agent': 'Mozilla/5.0'})
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(dest, 'wb') as f, tqdm(
            total=total, unit='B', unit_scale=True,
            desc=f'  ⬇️ {Path(dest).name}', leave=False
        ) as bar:
            for chunk in r.iter_content(8192):
                f.write(chunk)
                bar.update(len(chunk))
        return os.path.getsize(dest) > 0
    except Exception as e:
        print(f'    ❌ Direct download failed: {e}')
        if os.path.exists(dest): os.remove(dest)
        return False

def download_ytdlp(url, dest_dir, title):
    """Download video using yt-dlp (YouTube, Vimeo, etc.)"""
    safe = re.sub(r'[^\w\s-]', '', title)[:40]
    out  = os.path.join(dest_dir, f'{safe}.%(ext)s')
    cmd  = ['yt-dlp', '-f', 'bestvideo[height<=720]+bestaudio/best[height<=720]/best',
             '--merge-output-format', 'mp4', '-o', out,
             '--no-playlist', '--quiet', '--no-warnings', url]
    try:
        subprocess.run(cmd, timeout=300, check=True)
        found = glob.glob(os.path.join(dest_dir, f'{safe}.*'))
        return found[0] if found else None
    except Exception as e:
        print(f'    ⚠️ yt-dlp failed: {e}')
        return None

def smart_download_video(url, dest_dir, title):
    """Try yt-dlp first, then direct download."""
    # Try yt-dlp for YouTube/Vimeo etc.
    is_streaming = any(x in url for x in ['youtube', 'youtu.be', 'vimeo', 'dailymotion'])
    if is_streaming:
        path = download_ytdlp(url, dest_dir, title)
        if path and os.path.exists(path):
            return path
    # Direct download for CDN links
    ext  = url.split('.')[-1].split('?')[0][:4] or 'mp4'
    safe = re.sub(r'[^\w\s-]', '', title)[:30]
    dest = os.path.join(dest_dir, f'{safe}.{ext}')
    if download_direct(url, dest):
        return dest
    return None

print('✅ Download helpers ready.')

In [ ]:
# ─── Cell 5: Fetch Platform Data ─────────────────────────────────────────────

def api(path, params=None):
    try:
        r = requests.get(f'{API_BASE_URL}{path}', params=params or {}, timeout=30)
        if r.status_code == 200:
            return r.json()
        print(f'  ❌ API {path}: {r.status_code}')
    except Exception as e:
        print(f'  ❌ API error: {e}')
    return None

def fetch_all_content(semester_id):
    print(f'📡 Fetching all content for semester: {semester_id}')
    
    # Fetch modules
    raw = api('/api/v1/modules-all', {'semesterId': semester_id})
    if not raw:
        # Fallback: try mobile endpoint which returns full nested data
        raw = api('/api/v1/mobile/semester-content', {'semesterId': semester_id})
    
    modules = raw if isinstance(raw, list) else (raw or {}).get('modules', [])
    print(f'  ✅ Found {len(modules)} modules.')
    
    result = []
    for mod in modules:
        mod_id   = mod.get('id')
        mod_name = mod.get('name', 'Unknown')
        subjects = mod.get('subjects', [])
        
        # If subjects are not nested, fetch them
        if not subjects:
            subj_raw = api(f'/api/v1/modules/{mod_id}')
            subjects = (subj_raw or {}).get('subjects', [])
        
        full_subjects = []
        for sub in subjects:
            sub_id   = sub.get('id')
            sub_name = sub.get('name', 'Unknown')
            lectures = sub.get('lectures', [])
            
            # Fetch full lecture details if they don't have files
            full_lectures = []
            for lec in lectures:
                lec_id = lec.get('id')
                # Check if lecture already has file data
                if not (lec.get('files') or lec.get('videos') or lec.get('records')):
                    lec_detail = api(f'/api/v1/lectures/{lec_id}')
                    if lec_detail:
                        lec = {**lec, **lec_detail}
                full_lectures.append(lec)
            
            full_subjects.append({'id': sub_id, 'name': sub_name, 'lectures': full_lectures})
        
        total_lectures = sum(len(s.get('lectures', [])) for s in full_subjects)
        print(f'  📦 {mod_name}: {len(full_subjects)} subjects, {total_lectures} lectures')
        result.append({'id': mod_id, 'name': mod_name, 'subjects': full_subjects})
    
    return result

MODULES = fetch_all_content(SEMESTER_ID)
print(f'\n✅ Ready to sync {len(MODULES)} modules to Telegram Group.')

In [ ]:
# ─── Cell 6: Upload Engine ────────────────────────────────────────────────────

def upload_lecture(thread_id, lecture, lec_num, lec_dir):
    title   = lecture.get('title', f'Lecture {lec_num}')
    videos  = lecture.get('videos', [])
    records = lecture.get('records', [])
    files   = lecture.get('files', [])
    
    print(f'      📝 [{lec_num}] {title}')
    send_text(thread_id, f'📝 *{lec_num}. {title}*')
    
    # ── Videos ──
    for i, vid in enumerate(videos):
        name = vid.get('title', f'Video {i+1}')
        url  = vid.get('url', '')
        if not url: continue
        print(f'        🎬 {name}')
        path = smart_download_video(url, lec_dir, name)
        if path and os.path.exists(path):
            result = send_file(thread_id, path, caption=f'🎬 {name}', as_video=True)
            try: os.remove(path)
            except: pass
            if not result:
                send_text(thread_id, f'🎬 *{name}*\n[Watch]({url})')
        else:
            send_text(thread_id, f'🎬 *{name}*\n[Watch]({url})')
    
    # ── Records (Audio) ──
    for i, rec in enumerate(records):
        name = rec.get('title', f'Record {i+1}')
        url  = rec.get('url', '')
        if not url: continue
        print(f'        🎙 {name}')
        dest = os.path.join(lec_dir, f'rec_{i+1}.mp3')
        if download_direct(url, dest) and os.path.exists(dest):
            result = send_file(thread_id, dest, caption=f'🎙 {name}')
            try: os.remove(dest)
            except: pass
            if not result:
                send_text(thread_id, f'🎙 *{name}*\n[Listen]({url})')
        else:
            send_text(thread_id, f'🎙 *{name}*\n[Listen]({url})')
    
    # ── Files (PDFs, Docs) ──
    for i, fil in enumerate(files):
        name = fil.get('name', f'File {i+1}')
        url  = fil.get('url', '')
        mime = fil.get('mimeType', '')
        if not url: continue
        print(f'        📄 {name}')
        # Determine extension
        if 'pdf' in mime or url.lower().endswith('.pdf'):    ext = 'pdf'
        elif 'word' in mime or url.lower().endswith('.docx'): ext = 'docx'
        elif 'powerpoint' in mime:                            ext = 'pptx'
        else: ext = (url.split('?')[0].split('.')[-1] or 'bin')[:5]
        safe = re.sub(r'[^\w]', '_', name)[:30]
        dest = os.path.join(lec_dir, f'{safe}.{ext}')
        if download_direct(url, dest) and os.path.exists(dest):
            result = send_file(thread_id, dest, caption=f'📄 {name}')
            try: os.remove(dest)
            except: pass
            if not result:
                send_text(thread_id, f'📄 *{name}*\n[Download]({url})')
        else:
            send_text(thread_id, f'📄 *{name}*\n[Download]({url})')
    
    if not videos and not records and not files:
        send_text(thread_id, '_No attachments._')
    
    send_text(thread_id, '─' * 25)


def sync_module(module, color):
    name     = module['name']
    subjects = module.get('subjects', [])
    
    if name in SKIP_MODULES:
        print(f'  ⏭️ Skipping: {name}')
        return
    
    print(f'\n{"="*60}')
    print(f'📦 Module: {name}')
    
    thread_id = create_topic(name, color)
    if not thread_id:
        print(f'  ❌ Could not create topic. Skipping.')
        return
    
    time.sleep(1)
    
    total_lec = sum(len(s.get('lectures', [])) for s in subjects)
    send_text(thread_id,
        f'📦 *{name}*\n\n'
        f'📚 Subjects: {len(subjects)}\n'
        f'📝 Lectures: {total_lec}\n\n'
        f'_Synced automatically by MedOneX Bot 🤖_'
    )
    
    mod_dir = os.path.join(WORK_DIR, re.sub(r'[^\w]', '_', name)[:30])
    os.makedirs(mod_dir, exist_ok=True)
    
    for sub in subjects:
        sub_name = sub['name']
        lectures = sub.get('lectures', [])
        print(f'\n  📖 Subject: {sub_name} ({len(lectures)} lectures)')
        
        # Subject header with lecture index
        index = '\n'.join([f'  {i+1}. {l.get("title","Lecture")}' for i, l in enumerate(lectures)])
        send_text(thread_id, f'\n📖 *{sub_name}*\n{"─"*25}\n{index}')
        
        lec_dir = os.path.join(mod_dir, re.sub(r'[^\w]', '_', sub_name)[:20])
        os.makedirs(lec_dir, exist_ok=True)
        
        for i, lec in enumerate(lectures):
            upload_lecture(thread_id, lec, i+1, lec_dir)
    
    print(f'✅ Module "{name}" done!')

print('✅ Upload engine ready.')

In [ ]:
# ─── Cell 7: 🚀 RUN SYNC ──────────────────────────────────────────────────────
# Run this cell to start the sync process.
# It will create topics and upload content one module at a time.

COLORS = [0x6FB9F0, 0xFFD67E, 0xCB86DB, 0x8EEE98, 0xFF93B2, 0xFB6F5F]

print('🚀 Starting MedOneX → Telegram Group Sync')
print(f'   Modules: {len(MODULES)}')
print(f'   Group  : {GROUP_ID}')
print('='*60)

for idx, module in enumerate(MODULES):
    sync_module(module, COLORS[idx % len(COLORS)])
    time.sleep(2)

print('\n' + '='*60)
print('🎉 ALL DONE! Check your Telegram Group.')
print('='*60)

In [ ]:
# ─── Cell 8 (Optional): Sync ONE specific module only ────────────────────────
# Useful for re-running a single module without duplicating others

MODULE_NAME    = "CNS"          # Name of the module to sync
EXISTING_TOPIC_ID = None        # Set to thread_id if topic already exists (to avoid duplicate)

target = next((m for m in MODULES if MODULE_NAME.lower() in m['name'].lower()), None)
if target:
    if EXISTING_TOPIC_ID:
        # Upload directly to existing topic
        subjects = target.get('subjects', [])
        for sub in subjects:
            lec_dir = os.path.join(WORK_DIR, 'single_module')
            os.makedirs(lec_dir, exist_ok=True)
            send_text(EXISTING_TOPIC_ID, f'📖 *{sub["name"]}*')
            for i, lec in enumerate(sub.get('lectures', [])):
                upload_lecture(EXISTING_TOPIC_ID, lec, i+1, lec_dir)
    else:
        sync_module(target, 0x6FB9F0)
    print(f'✅ Done: {MODULE_NAME}')
else:
    print(f'❌ Module "{MODULE_NAME}" not found.')
    print('Available modules:', [m["name"] for m in MODULES])